<a href="https://colab.research.google.com/github/leandrobrana/Data-Science-II/blob/main/Final_ProyectoDS2Parte2Bra%C3%B1a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Análisis del comportamiento de la demanda eléctrica en el Sistema Argentino de Interconexión (SADI)**

El análisis del comportamiento de la demanda eléctrica, la potencia pico y las variables climáticas en el Sistema Argentino de Interconexión (SADI) resulta fundamental para comprender cómo opera el sistema energético frente a variaciones estacionales, térmicas y sociales. La disponibilidad de datos históricos detallados —que incluyen demanda total y regional, temperaturas medias por área, tipo de día y potencia pico— permite estudiar de manera profunda las dinámicas que determinan la carga eléctrica diaria del país.

Este proyecto se motiva en una necesidad real: el sistema eléctrico argentino se encuentra cada vez más exigido por eventos climáticos extremos, patrones de consumo cambiantes y urbanización acelerada. Entender cómo se comporta la demanda y qué factores explican los picos máximos es crítico para la planificación operativa, la gestión del riesgo, la inversión en infraestructura y el diseño de políticas energéticas basadas en evidencia.

La audiencia que puede beneficiarse de este análisis incluye:

Operadores del sistema eléctrico (como CAMMESA), que requieren estimaciones confiables de demanda y potencia pico para asegurar la estabilidad del sistema.

Ingenieros y analistas energéticos, interesados en desarrollar modelos predictivos y evaluar escenarios futuros.

Áreas de planificación y regulación, que necesitan comprender patrones estacionales y regionales para definir estrategias de expansión, eficiencia y resiliencia.

Equipos de ciencia de datos, que ven en esta base un caso sólido para aplicar técnicas de machine learning, imputación de datos faltantes, modelado predictivo y análisis multivariante.

El objetivo final es descubrir patrones, explicar comportamientos y construir modelos que permitan anticipar la demanda y los picos de potencia, maximizando la confiabilidad del sistema y facilitando la toma de decisiones en un contexto energético cada vez más desafiante.


# **Preguntas de investigación**

1 - ¿La demanda total del SADI aumenta significativamente cuando la temperatura media supera cierto umbral (olas de calor/frío)?

2 - ¿Qué regiones aportan mayor variabilidad diaria a la demanda total y cuáles son más sensibles a la temperatura?

3 - ¿Cómo se comporta la potencia pico en relación con la demanda total?

4 - ¿Hay alguna tendencia creciente o decreciente de la demanda a lo largo de los años?

5 - ¿Qué regiones muestran mayor variabilidad de demanda entre días laborables y fines de semana?

# **Instalaciones**

In [ ]:
!pip install unidecode
!pip install missingno

# **Import de librerias**

In [ ]:
import requests
import pandas as pd
import os
from unidecode import unidecode
from functools import reduce
import ipywidgets as widgets
from IPython.display import display
from google.colab import files
import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import numpy as np
from sklearn.model_selection import RandomizedSearchCV

# **Configuración de variables globales**

In [ ]:
# Fechas de inicio y fin utilizadas para el filtrado del dataset y la consulta a la API histórica de clima.
start_date = "2017-01-01"
end_date = "2025-12-31"

In [ ]:
# Se define array regiones, el cual va hacer utilizado en análisis posteriores
columnas_regiones = ['GRAN BS.AS.', 'BUENOS AIRES', 'CENTRO', 'LITORAL','CUYO', 'NORESTE', 'NOROESTE', 'COMAHUE', 'PATAGONICA']

In [ ]:
# Se definen umbrales de temperaturas máximas y mínimas para futuros análisis
umbral_calor = 25   # Umbral de ola de calor
umbral_frio = 15    # Umbral de ola de frío

# **Carga de datasets. Limpieza y transformación de datos**

In [ ]:
# Habilita la carga del dataset "Demanda Diaria MEM" desde el entorno local.
from google.colab import files
uploaded = files.upload()

In [ ]:
# Se extrae la hoja de interés para el presente análisis.
df_demanda = pd.read_excel("Demanda Diaria MEM.xlsx", sheet_name="Datos Región", skiprows=4)
# Se elimina la siguiente columna, ya que se utilizarán temperaturas de referencia por región.
df_demanda = df_demanda.drop(columns=["TEMPERATURA REFERENCIA MEDIA GBA °C"])
# Se eliminan caracteres especiales y se unifican los encabezados.
df_demanda.columns = [unidecode(col)        # Quita acentos
              .strip()                      # Quita espacios alrededor
              .upper()                      # Mayúsculas
              for col in df_demanda.columns]
# Se genera una nueva columna a partir de otras variables.
df_demanda['ESTACION'] = df_demanda['FECHA'].dt.month % 12 // 3 + 1
estaciones = {1: 'Verano', 2: 'Otoño', 3: 'Invierno', 4: 'Primavera'}
df_demanda['ESTACION'] = df_demanda['ESTACION'].map(estaciones)
# Se convierten las columnas de tipo object a tipo string.
df_demanda["TIPO DIA"] = df_demanda["TIPO DIA"].astype("string")
df_demanda["ESTACION"] = df_demanda["ESTACION"].astype("string")
# Se redondean a dos decimales todas las variables numéricas del dataset.
df_demanda = df_demanda.round(2)
# Se verifica que el dataset se procesó correctamente.
print("Dimensión:", df_demanda.shape)
display(df_demanda.head())
df_demanda.info()
display(df_demanda.isnull().sum())

In [ ]:
# Habilita la carga del dataset "Históricos valores de energía y Potencia" desde el entorno local.
from google.colab import files
uploaded = files.upload()

In [ ]:
# Se extrae la hoja de interés para el presente análisis.
df_generacion = pd.read_excel("Históricos valores de energía y Potencia.xlsx", sheet_name="Históricos Energía y Potencia", skiprows=5)
# Se extraen las columnas de interes.
df_generacion = df_generacion[["FECHA", "Potencia Pico SADI (MW)", "Hora Potencia Pico"]]
# Se eliminan caracteres especiales y se unifican los encabezados.
df_generacion.columns = [unidecode(col)               # Quita acentos
              .strip()                                # Quita espacios alrededor
              .upper()                                # Mayúsculas
              for col in df_generacion.columns]
# Se aplica un filtro de fecha para acotar los datos del segundo dataset.
df_generacion = df_generacion[(df_generacion["FECHA"] >= start_date) & (df_generacion["FECHA"] <= end_date)]
# Se redondean a dos decimales todas las variables numéricas del dataset.
df_generacion = df_generacion.round(2)
# Se verifica que el dataset se procesó correctamente.
print("Dimensión:", df_generacion.shape)
display(df_generacion.head())
df_generacion.info()
display(df_generacion.isnull().sum())

In [ ]:
# Se realiza el merge entre ambos datasets utilizando la columna "FECHA" como clave.
df_merge = pd.merge(df_demanda, df_generacion, on="FECHA", how="outer")
# Se verifica que los datasets se unieron correctamente
print("Dimensión:", df_merge.shape)
display(df_merge.head())
df_merge.info()
display(df_merge.isnull().sum())

In [ ]:
# Se utiliza una API para obtener datos históricos de temperatura por región.

# Se define el arreglo de regiones y coordenadas.
regiones = {
    "GRAN BS.AS.": (-34.61, -58.38),
    "BUENOS AIRES": (-34.92, -57.95),
    "CENTRO": (-31.42, -64.18),
    "LITORAL": (-32.95, -60.66),
    "CUYO": (-32.89, -68.84),
    "NOROESTE": (-24.78, -65.41),
    "NORESTE": (-27.45, -58.99),
    "COMAHUE": (-38.95, -68.06),
    "PATAGONICA": (-45.87, -67.48)
}

url = "https://archive-api.open-meteo.com/v1/archive"
df_temperaturas = pd.DataFrame()
for region, (lat, lon) in regiones.items():

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "daily": "temperature_2m_mean",
        "timezone": "America/Argentina/Buenos_Aires"
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()

    df_region = pd.DataFrame({
        "FECHA": data["daily"]["time"],
        region: data["daily"]["temperature_2m_mean"]
    })

    if df_temperaturas.empty:
        df_temperaturas = df_region
    else:
        df_temperaturas = df_temperaturas.merge(df_region, on="FECHA")

df_temperaturas["FECHA"] = pd.to_datetime(df_temperaturas["FECHA"])
df_temperaturas = df_temperaturas.rename(
    columns={region: f"TEMPERATURA MEDIA {region}" for region in regiones.keys()}
)

# Se verifica que el dataset se procesó correctamente.
print("Dimensión:", df_temperaturas.shape)
display(df_temperaturas.head())
df_temperaturas.info()
display(df_temperaturas.isnull().sum())

In [ ]:
# Se realiza el merge entre ambos datasets utilizando la columna "FECHA" como clave.
df_final = pd.merge(df_merge, df_temperaturas, on="FECHA")

# Se reordena el DataFrame para ubicar la columna "FECHA" en la primera posición
cols = df_final.columns.tolist()
cols.insert(0, cols.pop(cols.index("FECHA")))
df_final = df_final[cols]

display(df_final.head(10))

## **Descarga del dataset**

In [ ]:
# Nombre del archivo
file_name = "dataset_final.xlsx"

# Guardar el DataFrame en Excel
df_final.to_excel(file_name, index=False)

# Crear botón
button = widgets.Button(
    description="Descargar dataset en Excel",
    button_style="success",
    icon="download"
)

# Acción del botón
def download_excel(b):
    files.download(file_name)

button.on_click(download_excel)

display(button)

# **Análisis exploratorio de datos**

## **Informacion del dataset**

In [ ]:
# Se obtiene la dimensión del dataset (filas, columnas).
df_final.shape

In [ ]:
# Se visualiza un resumen de los tipos de datos del dataset.
df_final.info()

In [ ]:
# Se calculan estadísticas descriptivas de las variables numéricas del dataset.
df_final.describe()

## **Análisis de valores nulos**

In [ ]:
# Cantidad de datos null
df_final.isnull().sum()

In [ ]:
# Grafico de matriz de valores faltantes
msno.matrix(df_final, color=(0, 0.2, 0.8))

In [ ]:
# Gráfico de barras de valores faltantes por columna
msno.bar(df_final, color=(0, 0.2, 0.8))

Se detectaron 153 valores faltantes en las variables “Potencia Pico SADI (MW)” y “Hora Potencia Pico”. La ausencia de estos registros se debe a información incompleta en el dataset Históricos valores de energía y Potencia.

Dado que estas variables no son centrales en el presente análisis, en esta primera etapa se opta por excluir dichos registros del estudio. En una instancia posterior se evaluará e implementará una estrategia metodológica adecuada para la imputación de estos valores faltantes.

## **Análisis de duplicados**

In [ ]:
# Se obtienen las filas duplicadas
duplicados_completos = df_final[df_final.duplicated()]
print(f"Filas completamente duplicadas: {len(duplicados_completos)}")

In [ ]:
# Se obtienen las fechas duplicadas (independientemente de otras columnas)
duplicadas_fecha = df_final[df_final.duplicated(subset='FECHA')]
print(f"Fechas duplicadas: {len(duplicadas_fecha)}")

## **Análisis de valores outliers**

In [ ]:
Q1 = df_final['DEMANDA TOTAL'].quantile(0.25)
Q3 = df_final['DEMANDA TOTAL'].quantile(0.75)
IQR = Q3 - Q1

# Limites
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

# Filtrar outliers
outliers = df_final[(df_final['DEMANDA TOTAL'] < limite_inferior) | (df_final['DEMANDA TOTAL'] > limite_superior)]

print(f"Se encontraron {outliers.shape[0]} outliers en la demanda total.")

plt.figure(figsize=(10, 5))
sns.boxplot(x=df_final['DEMANDA TOTAL'])
plt.title('Boxplot de Demanda Total')
plt.show()

Se identificaron 49 valores atípicos en la demanda total, concentrados mayormente en el extremo superior de la distribución. Estos valores no parecen responder a errores de medición, sino a eventos de alta exigencia del sistema, probablemente asociados a condiciones climáticas extremas. Dado su relevancia operativa, se decidió mantenerlos en el análisis.

In [ ]:
Q1 = df_final['POTENCIA PICO SADI (MW)'].quantile(0.25)
Q3 = df_final['POTENCIA PICO SADI (MW)'].quantile(0.75)
IQR = Q3 - Q1

# Limites
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

# Filtrar outliers
outliers = df_final[(df_final['POTENCIA PICO SADI (MW)'] < limite_inferior) | (df_final['POTENCIA PICO SADI (MW)'] > limite_superior)]

print(f"Se encontraron {outliers.shape[0]} outliers en la demanda total.")

plt.figure(figsize=(10, 5))
sns.boxplot(x=df_final['POTENCIA PICO SADI (MW)'])
plt.title('Boxplot de Demanda Total')
plt.show()

Se identificaron 59 valores atípicos en la potencia pico del SADI, concentrados en el extremo superior de la distribución. Estos valores representan eventos de máxima exigencia del sistema, probablemente asociados a condiciones climáticas extremas. Lejos de considerarse errores, constituyen información crítica para el análisis de comportamiento y planificación del sistema eléctrico.

In [ ]:
outliers_por_region = {}

for region in columnas_regiones:
    Q1 = df_final[region].quantile(0.25)
    Q3 = df_final[region].quantile(0.75)
    IQR = Q3 - Q1

    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    outliers = df_final[(df_final[region] < limite_inferior) | (df_final[region] > limite_superior)]

    outliers_por_region[region] = outliers.shape[0]

df_outliers = pd.DataFrame(list(outliers_por_region.items()), columns=['REGION', 'CANTIDAD_OUTLIERS'])
df_outliers = df_outliers.sort_values(by='CANTIDAD_OUTLIERS', ascending=False)
display(df_outliers)

df_final[columnas_regiones].plot(kind='box', figsize=(14, 6), vert=False)
plt.title('Boxplots por Región')
plt.show()

Se observa una concentración significativa de valores atípicos en regiones con mayor variabilidad térmica, especialmente Patagonia. Estos valores no parecen corresponder a errores de medición sino a eventos extremos de demanda, probablemente asociados a condiciones climáticas severas. Por lo tanto, se decide conservarlos para el análisis principal, dado que representan el comportamiento real del sistema eléctrico.

## **Análisis de la variable objetivo**

In [ ]:
display(df_final['DEMANDA TOTAL'].describe())

# Histograma
plt.figure(figsize=(10, 5))
sns.histplot(df_final['DEMANDA TOTAL'], bins=30, kde=True, color='skyblue')
plt.title('Distribución de la demanda total diaria')
plt.xlabel('Demanda total (MW)')
plt.ylabel('Frecuencia')
plt.grid(True)
plt.show()

**Forma de la distribución:**

Es asimétrica positiva (sesgada a la derecha).

La mayoría de los valores se concentran entre los 13.000 y 17.000 MW.

Hay una “cola” hacia valores mayores, lo que indica presencia de días con altas demandas excepcionales.

**Asimetría**

La cola derecha más larga indica que existen días de alta demanda extrema posiblemente asociados a:

 - Olas de calor

 - Olas de frío

 - Días hábiles en invierno/verano

**Moda:**

El pico de frecuencia está cerca de los 14.500 MW, lo que sería la moda: el valor más frecuente de demanda diaria.

**Valores extremos (outliers):**

Hay valores que superan los 22.000 MW, lo que puede asociarse a días de alto consumo energético, probablemente en verano (uso de aire acondicionado) o invierno (calefacción).

**Interpretación:**

El sistema eléctrico nacional opera la mayor parte del tiempo en un rango bastante estable.

La asimetría sugiere que los picos de demanda (eventos extremos) son menos frecuentes pero relevantes.

# **Análisis de las preguntas de investigación**

## ¿La demanda total del SADI aumenta significativamente cuando la temperatura media supera cierto umbral (olas de calor/frío)?

In [ ]:

# Crear categoría según temperatura
df_final["CATEGORIA_TEMP"] = pd.cut(
    df_final["TEMPERATURA MEDIA GRAN BS.AS."],
    bins=[-50, umbral_frio, umbral_calor, 100],
    labels=["FRIO", "TEMPLADO", "CALOR"]
)

df_final.groupby("CATEGORIA_TEMP", observed=False)["DEMANDA TOTAL"].mean()

# Boxplot de demanda vs temperatura
sns.boxplot(
    data=df_final,
    x="CATEGORIA_TEMP",
    y="DEMANDA TOTAL",
    hue="CATEGORIA_TEMP",
    palette="viridis",
    legend=False
)

plt.title("Demanda Total según Categoría de Temperatura")
plt.show()

#Scatterplot para ver tendencia directa
sns.scatterplot(
    data=df_final,
    x="TEMPERATURA MEDIA GRAN BS.AS.",
    y="DEMANDA TOTAL",
    alpha=0.5
)
sns.regplot(
    data=df_final,
    x="TEMPERATURA MEDIA GRAN BS.AS.",
    y="DEMANDA TOTAL",
    scatter=False,
    color="red"
)
plt.title("Relación Temperatura vs Demanda Total")
plt.show()


El análisis evidencia un comportamiento no lineal entre temperatura media y demanda total del SADI. Se observa un mínimo de consumo en el rango templado, mientras que temperaturas extremas, tanto bajas como altas, generan incrementos significativos en la demanda. El efecto es particularmente marcado en condiciones de calor intenso, lo que sugiere una elevada sensibilidad del sistema ante olas de calor

## ¿Qué regiones aportan mayor variabilidad diaria a la demanda total y cuáles son más sensibles a la temperatura?

In [ ]:
variabilidad = pd.DataFrame({
    "REGION": columnas_regiones,
    "STD": [df_final[r].std() for r in regiones],
    "MEAN": [df_final[r].mean() for r in regiones]
})

variabilidad["CV"] = variabilidad["STD"] / variabilidad["MEAN"]
variabilidad = variabilidad.sort_values("CV", ascending=False)

print("Variabilidad diaria por región (ordenado por CV): \n")
print(variabilidad)


temp_cols = {
    "GRAN BS.AS.": "TEMPERATURA MEDIA GRAN BS.AS.",
    "BUENOS AIRES": "TEMPERATURA MEDIA BUENOS AIRES",
    "CENTRO": "TEMPERATURA MEDIA CENTRO",
    "LITORAL": "TEMPERATURA MEDIA LITORAL",
    "CUYO": "TEMPERATURA MEDIA CUYO",
    "NOROESTE": "TEMPERATURA MEDIA NOROESTE",
    "NORESTE": "TEMPERATURA MEDIA NORESTE",
    "COMAHUE": "TEMPERATURA MEDIA COMAHUE",
    "PATAGONICA": "TEMPERATURA MEDIA PATAGONICA",
}


sensibilidad = []

for region in columnas_regiones:
    temp_col = temp_cols[region]
    corr = df_final[[region, temp_col]].corr().iloc[0,1]
    sensibilidad.append((region, corr))

sensibilidad_df = pd.DataFrame(sensibilidad, columns=["REGION", "CORR_TEMP"])
sensibilidad_df = sensibilidad_df.sort_values("CORR_TEMP", ascending=False)

print("\nSensibilidad térmica por región (correlación): \n")
print(sensibilidad_df)


print("\nConclusiones\n")

region_mas_variable = variabilidad.iloc[0]["REGION"]
region_menos_variable = variabilidad.iloc[-1]["REGION"]

region_mas_sensible = sensibilidad_df.iloc[0]["REGION"]
region_menos_sensible = sensibilidad_df.iloc[-1]["REGION"]

print(f"- Región con MAYOR variabilidad diaria: {region_mas_variable}")
print(f"- Región con MENOR variabilidad diaria: {region_menos_variable}")
print(f"- Región MÁS sensible al clima (temperatura): {region_mas_sensible}")
print(f"- Región MENOS sensible al clima (temperatura): {region_menos_sensible}")

plt.figure(figsize=(12,5))
sns.barplot(
    data=variabilidad,
    x="REGION",
    y="CV",
    hue="REGION",       # requerido por seaborn
    palette="viridis",
    dodge=False,        # para que no repita barras
    legend=False        # para evitar leyenda duplicada
)
plt.xticks(rotation=45, ha="right")
plt.title("\n Ranking de Variabilidad Diaria por Región (Coeficiente de Variación)")
plt.ylabel("Coeficiente de Variación (CV)")
plt.xlabel("Región")
plt.tight_layout()
plt.show()




El análisis del coeficiente de variación evidencia que la región Noroeste presenta la mayor volatilidad diaria relativa, lo que sugiere una elevada sensibilidad ante cambios en las condiciones climáticas. En contraste, regiones como Comahue y Buenos Aires muestran perfiles más estables. La mayor variabilidad observada en Noroeste y Noreste podría estar asociada a una mayor dependencia de climatización eléctrica y a mayores amplitudes térmicas, lo que incrementa la elasticidad de la demanda frente a cambios de temperatura.

## ¿Cómo se comporta la potencia pico en relación con la demanda total?

In [ ]:
plt.scatter(df_final["DEMANDA TOTAL"], df_final["POTENCIA PICO SADI (MW)"])
plt.xlabel("Demanda Total")
plt.ylabel("Potencia Pico (MW)")
plt.title("Relación entre Demanda Total y Potencia Pico")
plt.show()

df_final["DEMANDA TOTAL"].corr(df_final["POTENCIA PICO SADI (MW)"])



df_final[["DEMANDA TOTAL", "POTENCIA PICO SADI (MW)"]].isna().sum()

df_model = df_final[["DEMANDA TOTAL", "POTENCIA PICO SADI (MW)"]].dropna()

import statsmodels.api as sm

X = sm.add_constant(df_model["DEMANDA TOTAL"])
y = df_model["POTENCIA PICO SADI (MW)"]

modelo = sm.OLS(y, X).fit()
print(modelo.summary())


Existe una relación lineal positiva muy fuerte entre ambas variables. La demanda total explica aproximadamente el 94% de la variabilidad de la potencia pico diaria, lo que indica que el nivel máximo del sistema está estrechamente vinculado al volumen agregado de consumo energético.

## ¿Hay alguna tendencia creciente o decreciente de la demanda a lo largo de los años?

In [ ]:
demanda_anual = df_final.groupby('ANO')['DEMANDA TOTAL'].mean()

plt.figure(figsize=(10, 6))
sns.lineplot(x=demanda_anual.index, y=demanda_anual.values, marker='o')
plt.title('Evolución de la demanda eléctrica promedio anual (2017–2025)')
plt.xlabel('Año')
plt.ylabel('Demanda total promedio (MW)')
plt.grid(True)
plt.tight_layout()
display(demanda_anual)
plt.show()

Se aprecia que la tendencia fue decreciente desde 2018 hasta 2020, coincidente con el arranque de la pandemia, y despues de esto se evidencia una tendencia creciente.

## ¿Qué regiones muestran mayor variabilidad de demanda entre días laborables y fines de semana?

In [ ]:
# Defino tipo de dia simple
df_final['TIPO_DIA_SIMPLE'] = df_final['TIPO DIA'].apply(lambda x: 'Finde' if x in ['Sabado o Semilaborable', 'Domingo o Feriado'] else 'Laborable')

# Dividir el dataframe según tipo de día
df_laborable = df_final[df_final['TIPO_DIA_SIMPLE'] == 'Laborable']
df_finde = df_final[df_final['TIPO_DIA_SIMPLE'] == 'Finde']

# Calcular la desviación estándar de cada grupo
std_laborable = df_laborable[columnas_regiones].std()
std_finde = df_finde[columnas_regiones].std()

variabilidad = pd.DataFrame({
    'Desv_Laborable': std_laborable,
    'Desv_Finde': std_finde
})
variabilidad['DIFERENCIA'] = abs(variabilidad['Desv_Laborable'] - variabilidad['Desv_Finde'])

# Ordenar por mayor diferencia
variabilidad_ordenada = variabilidad.sort_values(by='DIFERENCIA', ascending=False)


plt.figure(figsize=(10, 6))
sns.barplot(x=variabilidad_ordenada.index, y='DIFERENCIA', hue=variabilidad_ordenada.index, data=variabilidad_ordenada, palette='viridis', legend=False)
plt.xticks(rotation=45)
plt.title('\n Diferencia en la variabilidad de demanda entre días laborables y fines de semana')
plt.ylabel('Diferencia de Desviación Estándar (MW)')
plt.xlabel('Región')
plt.tight_layout()
display(variabilidad_ordenada)
plt.show()

El análisis de la desviación estándar evidencia que la región del Gran Buenos Aires presenta la mayor diferencia de variabilidad entre días laborables y fines de semana, superando ampliamente al resto de las regiones. Este comportamiento sugiere una fuerte influencia del componente industrial y comercial en la estructura de demanda semanal. En contraste, regiones como Noreste y Comahue muestran perfiles más estables, con menor sensibilidad al calendario.

# **Preprocesamiento de datos**

Se realizó un preprocesamiento de los datos que incluyó la codificación de variables categóricas mediante One-Hot Encoding, la división del dataset en conjuntos de entrenamiento y prueba, y la estandarización de variables numéricas utilizando parámetros ajustados exclusivamente sobre el conjunto de entrenamiento para evitar fuga de información.

Se identificó que la demanda total podía ser reconstruida a partir de la suma de las demandas regionales, lo que generaba un desempeño artificialmente elevado en los modelos.
Para evitar fuga de información, se eliminaron dichas variables, obteniendo resultados más representativos de la capacidad predictiva real.

In [ ]:
# Copiar dataset
df_process = df_final.copy()

# Eliminar variables que fugien información
columnas_a_eliminar = [
    "POTENCIA PICO SADI (MW)",
    "HORA POTENCIA PICO",
    "GRAN BS.AS.",
    "BUENOS AIRES",
    "CENTRO",
    "LITORAL",
    "CUYO",
    "NOROESTE",
    "NORESTE",
    "COMAHUE",
    "PATAGONICA"
]
df_process = df_process.drop(columns=columnas_a_eliminar, errors='ignore')

# Variable objetivo
target = "DEMANDA TOTAL"

# Separar X e y
X = df_process.drop(target, axis=1)
y = df_process[target]

# Eliminar datetime
X = X.select_dtypes(exclude=['datetime64[ns]'])

# Categóricas
categoricas = X.select_dtypes(include=['object', 'category', 'string']).columns

# One-Hot Encoding
X = pd.get_dummies(X, columns=categoricas, drop_first=True)

# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Variables numéricas
numericas = X_train.select_dtypes(include=['int64', 'float64']).columns

# Imputación (solo con train)
imputer = SimpleImputer(strategy='mean')

X_train[numericas] = imputer.fit_transform(X_train[numericas])
X_test[numericas] = imputer.transform(X_test[numericas])

# Escalado
scaler = StandardScaler()

X_train[numericas] = scaler.fit_transform(X_train[numericas])
X_test[numericas] = scaler.transform(X_test[numericas])

# Resultado
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

# **Entrenamiento y testeo**

Se utilizaron tres modelos de regresión y se prepararon los datos previamente según el tipo de variable. Luego, los modelos se evaluaron usando validación cruzada para asegurar que los resultados sean confiables.

In [ ]:
# Variable objetivo
target = "DEMANDA TOTAL"

X = df_process.drop(target, axis=1)
y = df_process[target]

# Eliminar datetime
X = X.select_dtypes(exclude=['datetime64[ns]'])

# Columnas por tipo
numericas = X.select_dtypes(include=['int64', 'float64']).columns
categoricas = X.select_dtypes(include=['object', 'category', 'string']).columns

# Preprocesador común
preprocesador = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ]), numericas),

    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
    ]), categoricas)
])

# Modelo 1: Regresión Lineal
pipeline_lr = Pipeline([
    ('preprocess', preprocesador),
    ('model', LinearRegression())
])

# Modelo 2: Random Forest
pipeline_rf = Pipeline([
    ('preprocess', preprocesador),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])

# Modelo 3: Gradient Boosting
pipeline_gb = Pipeline([
    ('preprocess', preprocesador),
    ('model', GradientBoostingRegressor(random_state=42))
])

# Validación cruzada
cv = KFold(n_splits=5, shuffle=True, random_state=42)

scores_lr = cross_val_score(pipeline_lr, X, y, cv=cv, scoring='r2')
scores_rf = cross_val_score(pipeline_rf, X, y, cv=cv, scoring='r2')
scores_gb = cross_val_score(pipeline_gb, X, y, cv=cv, scoring='r2')

# Resultados
print("Regresión Lineal:", np.mean(scores_lr), "+/-", np.std(scores_lr))
print("Random Forest:", np.mean(scores_rf), "+/-", np.std(scores_rf))
print("Gradient Boosting:", np.mean(scores_gb), "+/-", np.std(scores_gb))

Se evaluaron tres modelos de regresión (Lineal, Random Forest y Gradient Boosting) utilizando validación cruzada.
Los resultados muestran que la regresión lineal presenta un desempeño limitado, lo que indica que la relación entre las variables y la demanda eléctrica no es puramente lineal.
Por otro lado, los modelos basados en árboles, especialmente Gradient Boosting, lograron un mejor ajuste, alcanzando un coeficiente de determinación (R²) cercano a 0.89.
Esto evidencia la presencia de relaciones no lineales y complejas en los datos, siendo Gradient Boosting el modelo más adecuado para este problema.

La baja variabilidad observada en los resultados de la validación cruzada (reflejada en un bajo desvío estándar) indica que el modelo presenta un comportamiento estable y consistente frente a distintas particiones de los datos.

# **Optimización de modelos**

Se aplicó una técnica de optimización de hiperparámetros utilizando GridSearchCV sobre el modelo Random Forest, evaluando distintas combinaciones de parámetros mediante validación cruzada para mejorar el desempeño del modelo.

In [ ]:
param_grid = {
    'model__n_estimators': [100],
    'model__max_depth': [10, None]
}

random_search = RandomizedSearchCV(
    pipeline_rf,
    param_distributions=param_grid,
    n_iter=10,
    cv=cv,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)

random_search.fit(X, y)

print("Mejores parámetros:", random_search.best_params_)
print("Mejor R2:", random_search.best_score_)

Se aplicó una optimización de hiperparámetros mediante GridSearchCV sobre el modelo Random Forest. Si bien se logró una leve mejora en el coeficiente de determinación (R²), los resultados indican que el modelo base ya presentaba un buen desempeño, con un margen limitado de mejora mediante ajuste de parámetros.

A pesar de la optimización del modelo Random Forest, el modelo Gradient Boosting presentó un mejor desempeño general, por lo que se considera la alternativa más adecuada para el problema analizado.

# **Conclusiones**

En el presente trabajo se desarrolló un modelo de regresión para la predicción de la demanda eléctrica, utilizando un dataset que incluye variables temporales, geográficas y climáticas.

En una primera etapa, se realizó el preprocesamiento de los datos, que incluyó la limpieza de valores faltantes mediante imputación, la estandarización de variables numéricas y la codificación de variables categóricas. Asimismo, se identificaron y eliminaron variables altamente correlacionadas con la variable objetivo, evitando así problemas de fuga de información que podían generar resultados artificialmente elevados.

Posteriormente, se entrenaron y evaluaron tres modelos de regresión: Regresión Lineal, Random Forest y Gradient Boosting, utilizando validación cruzada para obtener métricas robustas y comparables. Los resultados evidenciaron que la Regresión Lineal presenta un desempeño limitado, lo que indica que la relación entre las variables explicativas y la demanda eléctrica no es puramente lineal.

Por otro lado, los modelos basados en árboles mostraron un mejor desempeño, destacándose especialmente Gradient Boosting, que alcanzó el mayor coeficiente de determinación (R² ≈ 0.89), junto con una baja variabilidad entre particiones, lo que indica un comportamiento estable y buena capacidad de generalización.

Adicionalmente, se aplicó una técnica de optimización de hiperparámetros sobre el modelo Random Forest mediante GridSearchCV. Si bien se obtuvo una leve mejora en su desempeño, los resultados indican que el modelo ya se encontraba cercano a su configuración óptima, y que el impacto del ajuste fue limitado en comparación con otros enfoques.

En conclusión, el modelo Gradient Boosting se posiciona como la mejor alternativa para el problema planteado, ya que logra capturar de manera más efectiva las relaciones no lineales presentes en los datos, proporcionando predicciones más precisas y consistentes. Este resultado resalta la importancia de seleccionar modelos adecuados a la naturaleza del problema y de realizar un adecuado preprocesamiento de los datos para obtener resultados confiables.